# Linear Algebra Deconvolution

Andrew E. Davidson aedavids@ucsc.edu 8/29/24  

Copyright (c) 2020-2023, Regents of the University of California All rights reserved.   https://polyformproject.org/licenses/noncommercial/1.0.0


CIBERSORTx is unable to deconvolve the serial dilution samples. Try relaxing the count & non-negative constraints. Solve using standard linear algebra

<span style="color:red;background-color:yellow">Linear Algebra solution does not work!</span>  

In [1]:
import ipynbname

import numpy as np
import pandas as pd

In [2]:
# notebookName = ipynbname.name()
# notebookPath = ipynbname.path()
# notebookDir = os.path.dirname(notebookPath)

# outDir = f'{notebookDir}/{notebookName}.out'
# imgOut = f'{outDir}/img'
# os.makedirs(imgOut, exist_ok=True) 
# print(f'imgOut:\n{imgOut}')

In [3]:
dataDir = "/private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10"
arithmeticDir = f"{dataDir}/bestArithemetic10Tempus.sh.out/cibersortInputDir"

In [4]:
bioMarkerPath = f"{arithmeticDir}/arithmeticBiomarkers_T1.csv"
print( f'{bioMarkerPath}' )
biomarkerDF = pd.read_csv(bioMarkerPath, sep="\t")
biomarkerDF

/private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10/bestArithemetic10Tempus.sh.out/cibersortInputDir/arithmeticBiomarkers_T1.csv


,"gene_id,arithmetic_diff"
0,"ENSG00000188536.13,1181604.1080667651"
1,"ENSG00000206172.8,114727.93120229256"
2,"(T)n,74022.28412666247"
3,"ENSG00000244734.4,70810.9106859027"
4,"(G)n,32096.473617957883"
5,"ENSG00000087086.15,31999.931449105945"
6,"ENSG00000210082.2,9422.67457209999"
7,"AluY,9239.72422822662"
8,"7SLRNA,8788.803008758954"
9,"AluSx1,8597.334548365066"


In [5]:
sigPath = f"{arithmeticDir}/signatureMatrix_T1.tsv"
print( f'{sigPath}' )
signatureDF = pd.read_csv(sigPath, sep="\t")
signatureDF

/private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10/bestArithemetic10Tempus.sh.out/cibersortInputDir/signatureMatrix_T1.tsv


,gene_id,SLDK3_T1_UD_S7_L007,SLDK3_T1_Control_S6_L007
0,ENSG00000188536.13,2.127554e+06,945950.043547
1,ENSG00000206172.8,2.780745e+05,163346.600693
2,(T)n,7.631708e+04,2294.799232
3,ENSG00000244734.4,4.725533e+05,543364.214188
4,(G)n,3.520969e+04,3113.217494
5,ENSG00000087086.15,1.268836e+04,44688.295512
6,ENSG00000210082.2,3.784704e+04,28424.368808
7,AluY,9.967946e+03,728.221353
8,7SLRNA,1.250147e+04,21290.268092
9,AluSx1,9.251499e+03,654.164945


In [6]:
mixturePath = f"{arithmeticDir}/mixtureMatrix_T1.tsv"
print( f'{mixturePath}' )
mixtureDF = pd.read_csv(mixturePath, sep="\t")
mixtureDF

/private/groups/kimlab/aedavids/deconvolution/tempus/best/bestArithemetic10/bestArithemetic10Tempus.sh.out/cibersortInputDir/mixtureMatrix_T1.tsv


,gene_id,SLDK3_T1_100_S1_L007,SLDK3_T1_100K_S2_L007,SLDK3_T1_10K_S3_L007,SLDK3_T1_1K_S4_L007,SLDK3_T1_1M_S5_L007,SLDK3_T1_Control_S6_L007,SLDK3_T1_UD_S7_L007
0,ENSG00000188536.13,523771.592747,789536.998285,523491.482228,661841.437440,891723.402554,945950.043547,2.127554e+06
1,ENSG00000206172.8,85277.551465,124777.118636,86561.790000,105771.781144,139885.190067,163346.600693,2.780745e+05
2,(T)n,1991.462728,5818.884686,2072.903092,1077.520369,1205.144023,2294.799232,7.631708e+04
3,ENSG00000244734.4,258057.380957,444358.691868,259841.450090,286485.007943,351011.911602,543364.214188,4.725533e+05
4,(G)n,1427.378458,2891.556427,1293.641913,2204.018937,1295.397100,3113.217494,3.520969e+04
5,ENSG00000087086.15,24942.334906,34626.139800,25627.325417,27166.574162,32645.068709,44688.295512,1.268836e+04
6,ENSG00000210082.2,18984.842000,27140.883935,19729.320852,19987.906816,24113.498467,28424.368808,3.784704e+04
7,AluY,8268.330883,3088.301504,8986.567511,3694.492746,546.827464,728.221353,9.967946e+03
8,7SLRNA,13527.122307,25167.471198,15912.194274,14075.950137,17326.821040,21290.268092,1.250147e+04
9,AluSx1,7145.612432,539.558467,8607.190359,3398.702841,642.389545,654.164945,9.251499e+03


In [7]:
signatureDF.loc[:, "SLDK3_T1_UD_S7_L007"] == mixtureDF.loc[:, "SLDK3_T1_UD_S7_L007"]

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
Name: SLDK3_T1_UD_S7_L007, dtype: bool

In [8]:
signatureDF.loc[:, "SLDK3_T1_Control_S6_L007"] == mixtureDF.loc[:, "SLDK3_T1_Control_S6_L007"]

0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
Name: SLDK3_T1_Control_S6_L007, dtype: bool

## Solve for fractions matrix

In [9]:
# select numeric cols
numericCols = ~signatureDF.columns.isin( ["gene_id"] )                                     
sigNP = signatureDF.loc[:, numericCols].values

# select numeric columns
numericCols = ~mixtureDF.columns.isin( ["gene_id"] )                                     
mixtureNP = mixtureDF.loc[:, numericCols].values

print(f' sigNP.shape : {sigNP.shape} mixtureNP.shape : {mixtureNP.shape}')

# Calculate the pseudo-inverse of the signature matrix
signaturePinvNP = np.linalg.pinv( sigNP )
print(f"\n signaturePinvNP.shape : {signaturePinvNP.shape}\n{signaturePinvNP}")

print("\n\n")

FTranspose = np.matmul( signaturePinvNP, mixtureNP)
print(f"Matrix FTranspose.shape :{FTranspose.shape}")
print(FTranspose)

print(f"\n\nF.shape :{FTranspose.shape}")
print(np.transpose( FTranspose) )

 sigNP.shape : (10, 2) mixtureNP.shape : (10, 7)

 signaturePinvNP.shape : (2, 10)
[[ 7.59455880e-07 -7.37201192e-08  1.65059911e-07 -1.28305268e-06
   6.72023917e-08 -1.65566072e-07 -3.70082191e-08  1.96922509e-08
  -6.40685306e-08  1.83714672e-08]
 [-6.66936391e-07  2.73890882e-07 -3.11586143e-07  2.88203175e-06
  -1.25073030e-07  3.51057424e-07  9.35809501e-08 -3.68008784e-08
   1.39123969e-07 -3.43532644e-08]]



Matrix FTranspose.shape :(2, 7)
[[ 5.54134008e-02  1.31600433e-02  5.25686194e-02  1.21588265e-01
   2.09446584e-01  1.98744245e-16  1.00000000e+00]
 [ 4.28829937e-01  8.04150323e-01  4.35066908e-01  4.25724372e-01
   4.70765682e-01  1.00000000e+00 -4.81532124e-16]]


F.shape :(2, 7)
[[ 5.54134008e-02  4.28829937e-01]
 [ 1.31600433e-02  8.04150323e-01]
 [ 5.25686194e-02  4.35066908e-01]
 [ 1.21588265e-01  4.25724372e-01]
 [ 2.09446584e-01  4.70765682e-01]
 [ 1.98744245e-16  1.00000000e+00]
 [ 1.00000000e+00 -4.81532124e-16]]


In [10]:

# select numeric columns
numericCols = ~mixtureDF.columns.isin( ["gene_id"] )

sampleIds= mixtureDF.columns[numericCols]
FDF = pd.DataFrame(
    FTranspose,
    columns = sampleIds, 
    index = ["UD", "Control", ]
)

FDF.transpose()

,UD,Control
SLDK3_T1_100_S1_L007,5.541340e-02,4.288299e-01
SLDK3_T1_100K_S2_L007,1.316004e-02,8.041503e-01
SLDK3_T1_10K_S3_L007,5.256862e-02,4.350669e-01
SLDK3_T1_1K_S4_L007,1.215883e-01,4.257244e-01
SLDK3_T1_1M_S5_L007,2.094466e-01,4.707657e-01
SLDK3_T1_Control_S6_L007,1.987442e-16,1.000000e+00
SLDK3_T1_UD_S7_L007,1.000000e+00,-4.815321e-16


In [11]:
FDF = FDF.transpose()
FDF

,UD,Control
SLDK3_T1_100_S1_L007,5.541340e-02,4.288299e-01
SLDK3_T1_100K_S2_L007,1.316004e-02,8.041503e-01
SLDK3_T1_10K_S3_L007,5.256862e-02,4.350669e-01
SLDK3_T1_1K_S4_L007,1.215883e-01,4.257244e-01
SLDK3_T1_1M_S5_L007,2.094466e-01,4.707657e-01
SLDK3_T1_Control_S6_L007,1.987442e-16,1.000000e+00
SLDK3_T1_UD_S7_L007,1.000000e+00,-4.815321e-16


In [12]:
FDF = FDF.round(decimals=4)
FDF

,UD,Control
SLDK3_T1_100_S1_L007,0.0554,0.4288
SLDK3_T1_100K_S2_L007,0.0132,0.8042
SLDK3_T1_10K_S3_L007,0.0526,0.4351
SLDK3_T1_1K_S4_L007,0.1216,0.4257
SLDK3_T1_1M_S5_L007,0.2094,0.4708
SLDK3_T1_Control_S6_L007,0.0000,1.0000
SLDK3_T1_UD_S7_L007,1.0000,-0.0000


In [13]:
byRow = 1
FDF['rowSum'] =FDF.loc[:, ['UD', 'Control']].sum(axis=byRow)
FDF

,UD,Control,rowSum
SLDK3_T1_100_S1_L007,0.0554,0.4288,0.4842
SLDK3_T1_100K_S2_L007,0.0132,0.8042,0.8174
SLDK3_T1_10K_S3_L007,0.0526,0.4351,0.4877
SLDK3_T1_1K_S4_L007,0.1216,0.4257,0.5473
SLDK3_T1_1M_S5_L007,0.2094,0.4708,0.6802
SLDK3_T1_Control_S6_L007,0.0000,1.0000,1.0000
SLDK3_T1_UD_S7_L007,1.0000,-0.0000,1.0000


In [14]:
FDF['%UD'] =  FDF['UD'] / FDF['rowSum'] * 100.0 
# print()
# print( FDF['Control'] / FDF['rowSum'] * 100.0)
FDF['%Control'] =  FDF['Control'] / FDF['rowSum'] * 100.0 

orderedIdx = ['SLDK3_T1_100_S1_L007', 'SLDK3_T1_1K_S4_L007', 'SLDK3_T1_10K_S3_L007',
             'SLDK3_T1_100K_S2_L007', 'SLDK3_T1_1M_S5_L007', 
             'SLDK3_T1_Control_S6_L007', 'SLDK3_T1_UD_S7_L007']

FDF.loc[orderedIdx, :]

,UD,Control,rowSum,%UD,%Control
SLDK3_T1_100_S1_L007,0.0554,0.4288,0.4842,11.441553,88.558447
SLDK3_T1_1K_S4_L007,0.1216,0.4257,0.5473,22.218162,77.781838
SLDK3_T1_10K_S3_L007,0.0526,0.4351,0.4877,10.785319,89.214681
SLDK3_T1_100K_S2_L007,0.0132,0.8042,0.8174,1.614876,98.385124
SLDK3_T1_1M_S5_L007,0.2094,0.4708,0.6802,30.785063,69.214937
SLDK3_T1_Control_S6_L007,0.0000,1.0000,1.0000,0.000000,100.000000
SLDK3_T1_UD_S7_L007,1.0000,-0.0000,1.0000,100.000000,-0.000000
